# Atividade 02 · Aprendizagem e retropropagação

**Nome**  
**Matrícula**  
**Prazo** 28/11/2026, às 23h59, no horário de Fortaleza.

Esta atividade é individual. Reproduza a retropropagação de uma rede 2–2–1 em NumPy, confira a diferenciação automática em TensorFlow/Keras e compare um passo de SGD nas duas implementações. Os dados e os pesos iniciais são os mesmos dos slides. Preencha os TODOs e use as células de conferência após cada seção.

| Requisito | Pontos |
|---|---:|
| NumPy · gradientes e atualização | 3 |
| TensorFlow/Keras · gradientes, comparação e atualização | 3 |
| Experimentos e interpretação | 3 |
| Entrega completa no prazo | 1 |
| **Total** | **10** |

Entregue o notebook executado, com as saídas e as respostas.

## Dados e pesos de partida

Cada linha de X é uma observação. Temos $x_1=1$, $x_2=2$ e $y=1$. Nas matrizes de pesos, as linhas correspondem às origens e as colunas aos destinos. Na passagem de ida, obtemos $A_1=XW_1+b_1$, $H=\tanh(A_1)$ e $z=HW_2+b_2$. A perda média é calculada diretamente a partir de `z`.

In [ ]:
import numpy as np

X = np.array([[1.0, 2.0]], dtype=np.float64)
y = np.array([[1.0]], dtype=np.float64)

# Uma linha por origem e uma coluna por destino em cada matriz de pesos.
W1 = np.array([[0.1, -0.2], [0.4, 0.3]], dtype=np.float64)
b1 = np.array([0.0, 0.1], dtype=np.float64)
W2 = np.array([[0.2], [-0.5]], dtype=np.float64)
b2 = np.array([0.05], dtype=np.float64)
parametros = (W1, b1, W2, b2)
nomes_parametros = ("W1", "b1", "W2", "b2")

np.set_printoptions(precision=8, suppress=True)

In [ ]:
def propagacao_numpy(entradas, parametros_rede):
    pesos_oculta, vies_oculta, pesos_saida, vies_saida = parametros_rede

    # Cada linha de A1 e H corresponde a uma observação.
    A1 = entradas @ pesos_oculta + vies_oculta
    H = np.tanh(A1)
    z = H @ pesos_saida + vies_saida

    # sigmoid(z) = exp(-log(1 + exp(-z))), escrita de forma estável.
    p = np.exp(-np.logaddexp(0.0, -z))
    return A1, H, z, p


def perda_numpy(alvos, logits):
    # logaddexp(0, z) calcula ln(1 + exp(z)) de forma estável.
    return np.mean(np.logaddexp(0.0, logits) - alvos * logits)


A1, H, z, p = propagacao_numpy(X, parametros)
perda_inicial = perda_numpy(y, z)
print(f"Logit {z.item():.5f}")
print(f"Probabilidade {p.item():.5f}")
print(f"Perda inicial {perda_inicial:.5f}")

## 01 · Retropropagação em NumPy

Calcule o sinal da saída e depois o sinal antes da tanh. Para cada neurônio oculto, o fator local da tanh é $1-h_j^2$. O peso $v_j$ usado nessa etapa é o peso da propagação direta.

$$
\frac{\partial L}{\partial z} = p-y,\qquad
\frac{\partial L}{\partial a_j}=\frac{\partial L}{\partial z}v_j(1-h_j^2),
\qquad
\frac{\partial L}{\partial w_{ij}}=x_i\frac{\partial L}{\partial a_j}.
$$

Para os pesos da saída, os fatores locais são as ativações $h_1$ e $h_2$. Para cada viés, o fator local é 1. Preencha uma matriz ou vetor de gradientes para cada item de `parametros`, na mesma ordem e com o mesmo formato.

In [ ]:
# TODO: calcule o sinal do logit e o sinal antes de cada tanh.
dz = None
dA1 = None

# TODO: calcule os gradientes. Use os pesos originais nesta etapa.
dW1 = None
db1 = None
dW2 = None
db2 = None
gradientes_np = (dW1, db1, dW2, db2)

In [ ]:
# Conferência da seção 01. A referência usa duas perdas para aproximar
# a derivada de cada parâmetro. Mantenha a célula como está.
if any(gradiente is None for gradiente in gradientes_np):
    print("Complete os gradientes da seção 01.")
else:
    epsilon = 1e-6
    for k, (nome, parametro, gradiente) in enumerate(zip(nomes_parametros, parametros, gradientes_np)):
        referencia = np.empty_like(parametro)
        for indice in np.ndindex(parametro.shape):
            mais = [valor.copy() for valor in parametros]
            menos = [valor.copy() for valor in parametros]
            mais[k][indice] += epsilon
            menos[k][indice] -= epsilon
            z_mais = propagacao_numpy(X, mais)[2]
            z_menos = propagacao_numpy(X, menos)[2]
            referencia[indice] = (perda_numpy(y, z_mais) - perda_numpy(y, z_menos)) / (2 * epsilon)
        assert np.shape(gradiente) == parametro.shape, f"Confira o formato de {nome}."
        np.testing.assert_allclose(gradiente, referencia, atol=1e-7, err_msg=nome)
    print("Os nove gradientes passaram na conferência por diferenças finitas.")

## 02 · Diferenciação automática com TensorFlow/Keras

O modelo Keras recebe os mesmos parâmetros. Dentro de `GradientTape`, calcule o logit e a perda. Em seguida, peça os gradientes em relação às variáveis treináveis. A perda espera o alvo primeiro e o logit depois.

Compare os gradientes com os do NumPy. Calcule a maior diferença absoluta entre todas as entradas correspondentes.

In [ ]:
import tensorflow as tf
from tensorflow import keras


def criar_modelo(parametros_rede):
    # A saída Dense é linear. O Keras recebe z diretamente na BCE.
    modelo = keras.Sequential([
        keras.Input(shape=(2,), dtype="float64"),
        keras.layers.Dense(2, activation="tanh", dtype="float64"),
        keras.layers.Dense(1, dtype="float64"),
    ])
    modelo.set_weights(parametros_rede)
    return modelo


bce = keras.losses.BinaryCrossentropy(from_logits=True, dtype="float64")
X_tf = tf.constant(X)
y_tf = tf.constant(y)


modelo = criar_modelo(parametros)
with tf.GradientTape() as tape:
    # TODO: execute a propagação direta e a perda dentro da fita.
    z_keras = None
    perda_keras = None

# TODO: obtenha os gradientes usando as variáveis treináveis do modelo.
gradientes_keras = None

# TODO: calcule o maior erro absoluto entre NumPy e Keras.
# Use .numpy() para converter um tensor de gradiente em um array.
maior_diferenca = None

In [ ]:
# Conferência da seção 02. Compare os dois cálculos no mesmo ponto.
if gradientes_keras is None or maior_diferenca is None or any(g is None for g in gradientes_np):
    print("Complete a seção 01 e os TODOs da seção 02.")
else:
    np.testing.assert_allclose(z_keras.numpy(), z, atol=1e-12)
    np.testing.assert_allclose(perda_keras.numpy(), perda_inicial, atol=1e-12)
    for numpy_grad, keras_grad in zip(gradientes_np, gradientes_keras, strict=True):
        np.testing.assert_allclose(numpy_grad, keras_grad.numpy(), atol=1e-12)
    referencia = max(
        np.max(np.abs(numpy_grad - keras_grad.numpy()))
        for numpy_grad, keras_grad in zip(gradientes_np, gradientes_keras)
    )
    np.testing.assert_allclose(maior_diferenca, referencia, atol=1e-12, rtol=0)
    print(f"Maior diferença absoluta {maior_diferenca:.3e}")

## 03 · Atualização e interpretação

Use $\eta=0{,}1$ e aplique $w_{ij,\mathrm{novo}}=w_{ij,\mathrm{atual}}-\eta\,\partial L/\partial w_{ij}$. Em NumPy, monte um novo conjunto de parâmetros. Em Keras, use SGD com as variáveis do modelo novo da célula abaixo. Esse modelo começa com os pesos originais, assim você pode reexecutar a célula para repetir o mesmo passo.

Calcule a nova perda com o mesmo exemplo nas duas implementações. Use as saídas para justificar as respostas.

In [ ]:
taxa = 0.1

# TODO: aplique a atualização em NumPy e calcule a perda nova.
parametros_novos = None
perda_nova_numpy = None

# A cópia começa com os mesmos pesos usados para obter os gradientes.
modelo_atualizado = criar_modelo(parametros)
sgd = keras.optimizers.SGD(learning_rate=taxa)

# TODO: aplique os gradientes da seção 02 às variáveis de `modelo_atualizado`.
# Depois, calcule a perda dessa rede no mesmo exemplo.
perda_nova_keras = None

In [ ]:
# Conferência da seção 03.
if any(item is None for item in (parametros_novos, perda_nova_numpy, perda_nova_keras)):
    print("Complete a atualização e o cálculo da perda nas duas versões.")
else:
    esperados = [
        inicial - taxa * gradiente
        for inicial, gradiente in zip(parametros, gradientes_np)
    ]
    for esperado, calculado, keras_param in zip(
        esperados, parametros_novos, modelo_atualizado.get_weights(), strict=True
    ):
        np.testing.assert_allclose(calculado, esperado, atol=1e-8)
        np.testing.assert_allclose(keras_param, esperado, atol=1e-8)
    perda_recalculada = perda_numpy(y, propagacao_numpy(X, parametros_novos)[2])
    np.testing.assert_allclose(perda_nova_numpy, perda_recalculada, atol=1e-8)
    np.testing.assert_allclose(perda_nova_keras, perda_recalculada, atol=1e-8)
    print(f"Perda inicial {perda_inicial:.5f}")
    print(f"Perda depois do passo {perda_nova_numpy:.5f}")

### Respostas

1. Quais foram as perdas antes e depois do passo? Use a mudança na probabilidade da classe correta para interpretar o resultado.
2. Qual foi a maior diferença absoluta entre os gradientes de NumPy e TensorFlow? O que ela indica?
3. Por que usar o valor atualizado de $v_1$ para calcular $\partial L/\partial w_{11}$ altera o resultado? Indique quais valores desse produto vieram da propagação direta original.